# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [2]:
class matrix:
    def __init__(self, a, b=None):
        if isinstance(a, int) and isinstance(b, int):
            if a <= 0 or b <= 0:
                raise ValueError
            self.data = []
            for i in range(a):
                row = []
                for j in range(b):
                    row.append(0.0)
                self.data.append(row)

        elif isinstance(a, list) and b is None:
            if len(a) == 0:
                raise ValueError
            if not all(isinstance(row, list) for row in a):
                raise TypeError
            row_len = len(a[0])
            if row_len == 0:
                raise ValueError
            for row in a:
                if len(row) != row_len:
                    raise ValueError

            self.data = []
            for row in a:
                new_row = []
                for val in row:
                    new_row.append(float(val))
                self.data.append(new_row)

        else:
            raise TypeError

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"

    def __str__(self):
        s = ""
        for row in self.data:
            s += str(row) + "\n"
        return s

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def to_list(self):
        result = []
        for row in self.data:
            result.append(row[:])
        return result

    def copy_from(self, other):
        if isinstance(other, matrix):
            if self.shape() != other.shape():
                raise ValueError
            self.data = other.to_list()
        elif isinstance(other, list):
            temp = matrix(other)
            if self.shape() != temp.shape():
                raise ValueError
            self.data = temp.to_list()
        else:
            raise TypeError

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.data[key]

        if isinstance(key, slice):
            return matrix(self.data[key])

        if isinstance(key, tuple):
            r, c = key

            if isinstance(r, int) and isinstance(c, int):
                return self.data[r][c]

            rows = self.data[r] if isinstance(r, slice) else [self.data[r]]

            result = []
            for row in rows:
                if isinstance(c, int):
                    result.append([row[c]])
                else:
                    result.append(row[c])

            return matrix(result)

        raise TypeError

    def __setitem__(self, key, value):
        i, j = key
        self.data[i][j] = float(value)

    def transpose(self):
        n, m = self.shape()
        result = matrix(m, n)
        for i in range(n):
            for j in range(m):
                result[j, i] = self[i, j]
        return result

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        result = []
        for i in range(len(self.data)):
            result.append([self.data[i][n]])
        return matrix(result)

    def block(self, n0, n1, m0, m1):
        result = []
        for i in range(n0, n1):
            row = []
            for j in range(m0, m1):
                row.append(self.data[i][j])
            result.append(row)
        return matrix(result)

    def scalarmul(self, c):
        n, m = self.shape()
        result = matrix(n, m)
        for i in range(n):
            for j in range(m):
                result[i, j] = self[i, j] * c
        return result

    def add(self, other):
        if not isinstance(other, matrix):
            raise TypeError
        if self.shape() != other.shape():
            raise ValueError

        n, m = self.shape()
        result = matrix(n, m)
        for i in range(n):
            for j in range(m):
                result[i, j] = self[i, j] + other[i, j]
        return result

    def sub(self, other):
        if not isinstance(other, matrix):
            raise TypeError
        if self.shape() != other.shape():
            raise ValueError

        n, m = self.shape()
        result = matrix(n, m)
        for i in range(n):
            for j in range(m):
                result[i, j] = self[i, j] - other[i, j]
        return result

    def mat_mult(self, other):
        if not isinstance(other, matrix):
            raise TypeError

        n1, m1 = self.shape()
        n2, m2 = other.shape()

        if m1 != n2:
            raise ValueError

        result = matrix(n1, m2)
        for i in range(n1):
            for j in range(m2):
                total = 0.0
                for k in range(m1):
                    total += self[i, k] * other[k, j]
                result[i, j] = total
        return result

    def element_mult(self, other):
        if not isinstance(other, matrix):
            raise TypeError
        if self.shape() != other.shape():
            raise ValueError

        n, m = self.shape()
        result = matrix(n, m)
        for i in range(n):
            for j in range(m):
                result[i, j] = self[i, j] * other[i, j]
        return result

    def equals(self, other):
        if not isinstance(other, matrix):
            return False
        if self.shape() != other.shape():
            return False

        n, m = self.shape()
        for i in range(n):
            for j in range(m):
                if self[i, j] != other[i, j]:
                    return False
        return True

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        elif isinstance(other, matrix):
            return self.mat_mult(other)
        else:
            raise TypeError

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)
        else:
            raise TypeError

    def __eq__(self, other):
        return self.equals(other)


def constant(n, m, c):
    result = matrix(n, m)
    for i in range(n):
        for j in range(m):
            result[i, j] = float(c)
    return result

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    result = matrix(n, n)
    for i in range(n):
        result[i, i] = 1.0
    return result

In [3]:
# TEST CELL (for checking)

A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print("A:")
print(A)

print("B:")
print(B)

print("A + B:")
print(A + B)

print("A - B:")
print(A - B)

print("A * B:")
print(A * B)

print("2 * A:")
print(2 * A)

print("Transpose of A:")
print(A.transpose())

print("Row 1 of A:")
print(A.row(1))

print("Column 0 of A:")
print(A.column(0))

print("Block of A:")
print(A.block(0, 2, 0, 1))

print("Identity matrix:")
print(eye(2))

print("Equality test:")
print(A == matrix([[1, 2], [3, 4]]))

A:
[1.0, 2.0]
[3.0, 4.0]

B:
[5.0, 6.0]
[7.0, 8.0]

A + B:
[6.0, 8.0]
[10.0, 12.0]

A - B:
[-4.0, -4.0]
[-4.0, -4.0]

A * B:
[19.0, 22.0]
[43.0, 50.0]

2 * A:
[2.0, 4.0]
[6.0, 8.0]

Transpose of A:
[1.0, 3.0]
[2.0, 4.0]

Row 1 of A:
[3.0, 4.0]

Column 0 of A:
[1.0]
[3.0]

Block of A:
[1.0]
[3.0]

Identity matrix:
[1.0, 0.0]
[0.0, 1.0]

Equality test:
True
